# A4 — Policy learning y uplift: política óptima bajo presupuesto (IHDP)

**Curso:** Sistema de Inteligencia Causal · *profesionales*  
**Sesiones:** S22–S23 · **Módulo 4**  
**Lectura monografía:** Cap. **21** Policy learning · Cap. **22** Uplift · Cap. **25** Decisiones (contexto)  
**Labs:** L21 · L22 · L23  
**Dataset:** `real/ihdp` (continúa A3)  
**Técnica:** top-k / umbral bajo presupuesto, welfare, Qini, baselines, equidad  
**Alimenta Serie B:** **B3** (`artifacts/A4/policy.joblib` + servicio local)

### Pregunta de valor (disrupción)

> *Con capacidad para tratar solo el 20%, ¿a quién asignamos el cupo?*  
> *Compara política causal vs aleatoria vs “tratar al de peor outcome predicho”.*

### Objetivos de aprendizaje

1. Convertir CATE en política $\pi(x)$ bajo presupuesto $B$.
2. Evaluar welfare (IPW sin truth; oracle $\tau$ solo al final).
3. Comparar baselines **aleatorio** y **outcome-only** (no causal).
4. Auditar equidad simple de la política.
5. Serializar política + *policy card* para B3.


### Decisión de negocio

> *Si solo predijéramos el outcome, el presupuesto se asignaría a quienes tienen peor pronóstico ("outcome-only") — el error clásico de "tratar al más enfermo".*  
> *Con el efecto (política top-k sobre CATE), la decisión de negocio es distinta: el 20% de capacidad se asigna a quien más se beneficia de la intervención, medido en welfare, no en necesidad aparente.*

La comparación (§5: `cate_topk` vs `random` vs `outcome_only`) exige reglas congeladas con `train` + `validation` antes de evaluar `test`; nunca se elige un ganador con ese holdout. Como este test ya fue inspeccionado por la versión anterior, incluso una nueva ejecución congelada es **exploratoria/contaminada**, no evidencia confirmatoria.

## Aviso sobre los datos (transparencia)

Este notebook usa un escenario del **Causal Benchmark Ecosystem**.

Los archivos en `datasets/real/` **no son datos clínicos ni administrativos reales**.  
Lo auténtico de la literatura son **nombres de variables estilo Hill (IHDP), diseño del estudio y órdenes de magnitud de referencia**.  
Los valores numéricos se generan con SEM / fórmulas documentadas para que exista **ground truth** (`truth_*.csv`) usable en evaluación y en el evaluación docente.

| Capa | Archivos | Quién la ve en este notebook |
|------|----------|------------------------------|
| Observables | `observed_*.csv` | **Siempre** (flujo del estudiante) |
| Ground truth | `truth_*.csv` (`y0`, `y1`, `tau`) | **Solo** sección final marcada «evaluación con ground truth / welfare oracle» |
| Identificación / scorecard | `identification.json`, `metadata.json`, `scorecard.json` | Lectura guiada |
| Artefactos A3 | `artifacts/A3/cate_scores_test.csv` | Referencia exploratoria test-only; no basta para calibrar el protocolo congelado |

Consulta `metadata.json` → campos `what_is_real` / `what_is_synthetic`.

**Política de truth:** no uses `y0` / `y1` / `tau` fuera de la sección final de evaluación.


## 1. Setup y carga del escenario

Semilla global del curso: `RANDOM_STATE = 42`.  
Splits oficiales con roles fijos: **`train`** ajusta modelos; **`validation`** calibra umbrales/presupuestos; **`test`** evalúa una vez las reglas congeladas. No re-particionamos ni reelegimos con test.  
Presupuesto fijado por el curso antes de test: \(B = 0.20\) (tratar aproximadamente al top 20% por \(\hat\tau\)).

Se ajusta un T-learner fijo en `train` para disponer del **mismo scorer** en validation y test. El CSV test-only de A3 no contiene scores de validation de ese scorer y, por tanto, no se reutiliza para calibrar.

**Estado de la evidencia (`EVIDENCE_STATUS`):** cualquier resultado obtenido con la versión que calibraba umbrales o baselines sobre test, y cualquier reevaluación de este mismo holdout ya inspeccionado, es **exploratorio/contaminado**. El protocolo queda preespecificado para esta lección docente, pero eso no equivale a un preregistro prospectivo. Editar o volver a ejecutar el notebook no repara la contaminación; una afirmación confirmatoria exigiría un holdout nuevo no inspeccionado y un protocolo prospectivo.


In [ ]:
# Paths relativos al monorepo (kernel-less: funciona con nbconvert/pytest)
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_NOTEBOOK_DIR = Path.cwd()
_CANDIDATES = [
    _NOTEBOOK_DIR.parent,  # .../03_notebooks_caso_estudio if cwd is notebooks/
    _NOTEBOOK_DIR,         # if cwd is already package root
]
for _p in _CANDIDATES:
    if _p is None:
        continue
    if (_p / "lib" / "paths.py").is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        PACKAGE_ROOT = _p
        break
else:
    here = Path.cwd().resolve()
    PACKAGE_ROOT = None
    for parent in [here, *here.parents]:
        if (parent / "lib" / "paths.py").is_file():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            PACKAGE_ROOT = parent
            break
    if PACKAGE_ROOT is None:
        raise RuntimeError("Could not locate package root (lib/paths.py)")

from lib.paths import (
    RANDOM_STATE,
    SCENARIO_A4,
    artifacts_dir,
    ensure_dir,
    package_root,
    relative_to_repo,
)
from lib.data_io import (
    assert_no_truth_columns,
    load_scenario,
)
from lib.metrics import qini_auc, qini_curve, uplift_curve
from lib.plotting import apply_course_style, plot_qini, plot_uplift_bins
from lib.policy_utils import (
    apply_policy,
    build_frozen_policy,
    build_rich_policy_card,
    compare_policies,
    equity_table,
    fit_frozen_policy_calibration,
    load_policy,
    observed_policy_value,
    random_policy,
    save_policy,
)

assert RANDOM_STATE == 42
SCENARIO = SCENARIO_A4  # real/ihdp
BUDGET_B = 0.20
METHOD_SELECTED = "t_learner"  # scorer fijo del curso
CALIBRATION_ROLE = "validation"
EVALUATION_ROLE = "test"
EVIDENCE_STATUS = "exploratory_contaminated_holdout"
ART_DIR = ensure_dir(artifacts_dir("A4"))

print("PACKAGE_ROOT :", PACKAGE_ROOT)
print("SCENARIO     :", SCENARIO)
print("RANDOM_STATE :", RANDOM_STATE)
print("BUDGET_B     :", BUDGET_B)
print("ART_DIR      :", ART_DIR)
print("roles        : train → validation (calibrar) → test (evaluar una vez)")
print("evidence     :", EVIDENCE_STATUS)


In [ ]:
# Carga SOLO observables + sidecars (sin truth)
bundle = load_scenario(SCENARIO)
assert_no_truth_columns(bundle.train, context="train")
assert_no_truth_columns(bundle.validation, context="validation")
assert_no_truth_columns(bundle.test, context="test")

meta = bundle.meta
ident = bundle.identification
scorecard = bundle.scorecard

print("scenario_id :", bundle.scenario_id)
print("shapes      : train", bundle.train.shape,
      "| val", bundle.validation.shape,
      "| test", bundle.test.shape)
print("description :", meta.get("description") or meta.get("full_name"))
print("what_is_real     :", meta.get("what_is_real"))
print("what_is_synthetic:", meta.get("what_is_synthetic"))
print("ate_true (metadata, no usar en estimación):", meta.get("ate_true"))


In [ ]:
# Roles de columnas y matrices de trabajo
Y_COL = bundle.outcome_column()    # outcome
T_COL = bundle.treatment_column()  # treatment
X_COLS = bundle.covariate_columns()

print(f"Y = {Y_COL!r}  |  T = {T_COL!r}")
print(f"X ({len(X_COLS)}): {X_COLS}")

Y_tr, T_tr, X_tr = bundle.ytx("train")
Y_va, T_va, X_va = bundle.ytx("validation")
# Test se carga con el bundle para validar esquema, pero sus Y/T/X no se
# extraen ni se usan para ajustar, calibrar o seleccionar antes de §5.

print("train treatment rate:", float(T_tr.mean()))
assert list(X_tr.columns) == X_COLS


## 2. Ajustar el scorer en train y puntuar validation

`fit_frozen_policy_calibration` ajusta el T-learner fijo exclusivamente en `train` y devuelve scores de **validation** más un **scorer picklable** para `policy.predict(X=...)` en B3.  
Esta celda no puntúa test. El CSV test-only de A3 no sirve como sustituto: mezclar un scorer en calibration y otro en test invalidaría la correspondencia del protocolo.


In [ ]:
tau_cal, method_name, scorer = fit_frozen_policy_calibration(
    SCENARIO,
    method_selected=METHOD_SELECTED,
    calibration_role=CALIBRATION_ROLE,
    evaluation_role=EVALUATION_ROLE,
    random_state=RANDOM_STATE,
)

print("method_name         :", method_name)
print("n validation scores :", len(tau_cal))
print("mean tau_cal        :", float(np.mean(tau_cal)))
print("std  tau_cal        :", float(np.std(tau_cal)))
assert len(tau_cal) == len(X_va), "CATE scores must align with validation rows"
assert np.isfinite(tau_cal).all()


## 3. De CATE a política bajo presupuesto \(B\)

Regla miópica (top-k / umbral calibrado):

\[
\lambda = \mathrm{Quantile}_{1-B}\big(\hat\tau(x)\big), \quad
\pi(x) = \mathbf{1}\{\hat\tau(x) \ge \lambda\}.
\]

El cuantil se calcula **solo con scores de validation**. Con \(B=0.20\) preespecificado, la regla busca tratar aproximadamente al 20% con mayor uplift estimado; la tasa realizada en test puede diferir.
La sensibilidad a \(B \in \{0.10, 0.20, 0.30\}\) también usa únicamente validation. No se evalúan esos presupuestos alternativos en test ni se elige un ganador con resultados de test.


In [ ]:
# Calibrar y congelar la política con validation; B está preespecificado.
policy = build_frozen_policy(
    tau_cal,
    calibration_role=CALIBRATION_ROLE,
    evaluation_role=EVALUATION_ROLE,
    budget_fraction=BUDGET_B,
    mode="topk",
    method_name=method_name,
    feature_names=X_COLS,
    score_model=scorer,
    meta={
        "scenario": SCENARIO,
        "scenario_id": bundle.scenario_id,
        "budget_fraction": BUDGET_B,
        "method_selected": method_name,
        "random_state": RANDOM_STATE,
    },
)

# Sensibilidad descriptiva SOLO en validation: no selecciona B con test.
rows_b = []
for b in (0.10, 0.20, 0.30):
    policy_b = build_frozen_policy(
        tau_cal,
        calibration_role=CALIBRATION_ROLE,
        evaluation_role=EVALUATION_ROLE,
        budget_fraction=b,
        mode="topk",
    )
    d_cal = apply_policy(tau_cal, policy_b)
    rows_b.append({"B": b, "threshold": policy_b.threshold, "validation_treat_rate": float(d_cal.mean()), "validation_n_treated": int(d_cal.sum())})
print("--- Sensibilidad en validation (no es evaluación test) ---")
print(pd.DataFrame(rows_b).to_string(index=False))

# Todavía no se puntúa test: primero se congelan también los baselines.
print(f"mode                 : {policy.mode}")
print(f"threshold λ (val)    : {policy.threshold:.6f}")
print(f"budget B (pre fijado): {policy.budget_fraction}")
print(f"policy_frozen        : {policy.meta['policy_frozen']}")


## 4. Ajustar y congelar los baselines antes de abrir test

El modelo outcome-only se ajusta en `train`; sus umbrales low/high se calculan solo en `validation`. La regla aleatoria tiene presupuesto y semilla preespecificados.  
Al terminar esta celda, CATE y todos los baselines están congelados. Todavía no se calculan scores ni outcomes de evaluación sobre test.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression

# Modelo factual E[Y|X]: ajuste en train; scores de validation para calibrar.
risk_model = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
risk_model.fit(np.asarray(X_tr, dtype=float), np.asarray(Y_tr, dtype=float))
y_risk_cal = risk_model.predict(np.asarray(X_va, dtype=float))

# Congelar ambos baselines outcome-only con umbrales de validation.
risk_low_policy = build_frozen_policy(
    -y_risk_cal,
    calibration_role=CALIBRATION_ROLE,
    evaluation_role=EVALUATION_ROLE,
    budget_fraction=BUDGET_B,
    mode="topk",
    method_name="outcome_only_low_y",
    feature_names=X_COLS,
)
risk_high_policy = build_frozen_policy(
    y_risk_cal,
    calibration_role=CALIBRATION_ROLE,
    evaluation_role=EVALUATION_ROLE,
    budget_fraction=BUDGET_B,
    mode="topk",
    method_name="outcome_only_high_y",
    feature_names=X_COLS,
)
# Aleatorio tiene B y semilla preespecificados; sus decisiones se generan
# al aplicar la regla en §5, sin calibrar con outcomes ni covariables test.

# La propensidad también se ajusta en train, antes de la evaluación test.
prop_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
prop_clf.fit(np.asarray(X_tr, dtype=float), np.asarray(T_tr, dtype=int))
print("CATE y baselines congelados; listos para una única evaluación test.")


## 5. Evaluación única en test de las reglas congeladas

Solo ahora se extraen `Y_te`, `T_te`, `X_te`, se puntúa test y se consultan sus outcomes. La tabla IPW, Qini y uplift son análisis preespecificados en este protocolo docente y no alimentan otra ronda de calibración o selección. Es una única evaluación **dentro de esta ejecución**, no una primera apertura histórica: por la inspección previa del holdout, los resultados conservan la etiqueta exploratorio/contaminado y no son confirmatorios.

| Política | Idea |
|----------|------|
| **cate_topk** | Tratar top-\(B\) por \(\hat\tau(x)\) (causal) |
| **random** | Tratar \(B\) al azar |
| **outcome_only_low_y** | Tratar a quienes tienen **peor** outcome predicho (necesidad / riesgo no causal) |

El baseline outcome-only es el error clásico de *tratar al más enfermo* sin modelar el **efecto** del tratamiento. Qini describe el ranking del scorer congelado y no demuestra identificación de \(\tau(x)\).


In [ ]:
# Abrir la matriz de evaluación solo después de congelar modelos y reglas.
Y_te, T_te, X_te = bundle.ytx(EVALUATION_ROLE)
dec_random = random_policy(len(X_te), BUDGET_B, random_state=RANDOM_STATE)

# Aplicar CATE una única vez con el scorer y umbral congelados.
tau_te = scorer.predict(X_te)
assert len(tau_te) == len(X_te) and np.isfinite(tau_te).all()
pred = policy.predict(scores=tau_te)
decision = pred["decision"]
treat_rate = float(np.mean(decision))
print(f"treat_rate π (test): {treat_rate:.4f}")
print(f"n_treated test     : {int(decision.sum())} / {len(decision)}")

# Aplicar las reglas outcome-only ya congeladas a test, sin reselección.
y_risk_te = risk_model.predict(np.asarray(X_te, dtype=float))
dec_risk_low = apply_policy(-y_risk_te, risk_low_policy)
dec_risk_high = apply_policy(y_risk_te, risk_high_policy)

# Aplicar el scorer de propensidad ajustado en train antes de abrir test.
e_te = prop_clf.predict_proba(np.asarray(X_te, dtype=float))[:, 1]

policies = {
    "cate_topk": decision,
    "random": dec_random,
    "outcome_only_low_y": dec_risk_low,
    "outcome_only_high_y": dec_risk_high,
}

cmp_ipw = compare_policies(
    policies,
    y=Y_te,
    t=T_te,
    propensity=e_te,
    evaluation_role=EVALUATION_ROLE,
)
factual_value_te = observed_policy_value(Y_te)
print("--- Frozen target-policy comparison (Horvitz–Thompson IPW; test) ---")
print(cmp_ipw.to_string(index=False))
print(f"Observed factual behavior-policy value mean(Y_test): {factual_value_te:.4f}")
print(
    "\nInterpretación: target_value_ipw estima V(π)=E[Y(π(X))] con pesos "
    "1{T=π(X)}/P(T=π(X)|X) y denominador n. No es la media factual "
    "mean(Y), ni una media condicionada a alineación. No seleccionar un "
    "ganador ni recalibrar después de ver esta tabla."
)

# Diagnósticos preespecificados del ranking, sin reselección posterior.
apply_course_style()
curve = qini_curve(Y_te, T_te, tau_te)
q_auc = qini_auc(curve)
uplift_df = uplift_curve(Y_te, T_te, tau_te, n_bins=10)
print(f"Qini AUC (test congelado): {q_auc:.4f}")
print(uplift_df.head().to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.2))
plot_qini(curve, ax=axes[0], title=f"Qini (test) — {method_name}")
plot_uplift_bins(uplift_df, ax=axes[1], title="Uplift by score bin (test)")
fig.suptitle(f"A4 frozen test diagnostics | B={BUDGET_B:.0%}", fontsize=11)
fig.tight_layout()
uplift_png = ART_DIR / "uplift_curve.png"
fig.savefig(uplift_png, dpi=120)
print("wrote", uplift_png)
plt.show()


## 6. Análisis de equidad (descriptivo)

Las covariables del escenario están en escala estandarizada del SEM.  
Auditamos **tasa de tratamiento bajo \(\pi\)** por:

- terciles de `bw` (birth weight proxy),
- split por mediana de `sex`,
- split por mediana de `b.marr`.

Este reporte descriptivo usa la política ya congelada; sus cortes de estratos describen test, pero no calibran la política ni se usan para cambiar la regla o escoger otro ganador. **No** certifica fairness legal ni ausencia de disparidad injusta; es un control de sanidad para discutir concentración de beneficios.


In [ ]:
eq_frame = X_te.copy()
eq_frame["bw_tercile"] = pd.qcut(
    eq_frame["bw"],
    q=3,
    labels=["T1_low", "T2_mid", "T3_high"],
    duplicates="drop",
)
sex_med = float(np.median(eq_frame["sex"]))
eq_frame["sex_bin"] = np.where(eq_frame["sex"] >= sex_med, "sex_high", "sex_low")
bm_med = float(np.median(eq_frame["b.marr"]))
eq_frame["bmarr_bin"] = np.where(eq_frame["b.marr"] >= bm_med, "b.marr_high", "b.marr_low")

eq_bw = equity_table(decision, eq_frame["bw_tercile"], group_name="bw_tercile")
eq_sex = equity_table(decision, eq_frame["sex_bin"], group_name="sex_bin")
eq_bm = equity_table(decision, eq_frame["bmarr_bin"], group_name="bmarr_bin")

print("--- treat rate by bw tercile ---")
print(eq_bw.to_string(index=False))
print("--- treat rate by sex median split ---")
print(eq_sex.to_string(index=False))
print("--- treat rate by b.marr median split ---")
print(eq_bm.to_string(index=False))

# Tabla exportable unificada
parts = []
for gname, tbl in (
    ("bw_tercile", eq_bw),
    ("sex_median_split", eq_sex.rename(columns={"sex_bin": "group"}) if "sex_bin" in eq_sex.columns else eq_sex),
    ("b.marr_median_split", eq_bm.rename(columns={"bmarr_bin": "group"}) if "bmarr_bin" in eq_bm.columns else eq_bm),
):
    p = tbl.copy()
    # normalize
    for c in list(p.columns):
        if c not in ("n", "n_treated", "treat_rate", "group"):
            p = p.rename(columns={c: "group"})
    if "group" not in p.columns:
        p = p.rename(columns={p.columns[0]: "group"})
    p.insert(0, "group_var", gname)
    parts.append(p[["group_var", "group", "n", "n_treated", "treat_rate"]])

equity_out = pd.concat(parts, ignore_index=True)
equity_path = ART_DIR / "equity_table.csv"
equity_out.to_csv(equity_path, index=False)
print("wrote", equity_path)
print(equity_out.to_string(index=False))


## 7. Export: `policy.joblib` + policy card (artefacto B3)

Interfaz contractual para Serie B:

```python
out = policy.predict(X=X_new)  # o policy.predict(scores=s)
# out["decision"], out["uplift_score"]
```

El policy card documenta alcance, features, umbral, supuestos, riesgos y **no-claims**.


In [ ]:
# Serializar la política y sus roles; no se vuelve a ajustar ni a evaluar.
card_md = build_rich_policy_card(
    scenario_id=bundle.scenario_id,
    scenario=SCENARIO,
    budget_fraction=BUDGET_B,
    method_name=method_name,
    threshold=float(policy.threshold),
    mode=str(policy.mode),
    feature_names=X_COLS,
    treat_rate_test=treat_rate,
    n_test=int(len(tau_te)),
    calibration_role=CALIBRATION_ROLE,
    evaluation_role=EVALUATION_ROLE,
    equity_group="bw_tercile",
    random_state=RANDOM_STATE,
)

policy_path = ART_DIR / "policy.joblib"
save_policy(policy, policy_path, policy_card=card_md)
print("wrote", policy_path)
print("wrote", ART_DIR / "policy_card.md")

# Registrar la única evaluación ya realizada; no llamar de nuevo al exportador.
export_info = {
    "policy_path": relative_to_repo(policy_path),
    "policy_card_path": relative_to_repo(ART_DIR / "policy_card.md"),
    "uplift_curve_path": relative_to_repo(uplift_png),
    "equity_table_path": relative_to_repo(equity_path),
    "scenario": SCENARIO,
    "scenario_id": bundle.scenario_id,
    "method_selected": method_name,
    "fit_role": "train",
    "calibration_role": CALIBRATION_ROLE,
    "evaluation_role": EVALUATION_ROLE,
    "policy_frozen_before_test": True,
    "evidence_status": EVIDENCE_STATUS,
    "budget_fraction": BUDGET_B,
    "threshold": float(policy.threshold),
    "treat_rate_test": treat_rate,
    "qini_auc_test": float(q_auc),
    "n_test": int(len(tau_te)),
    "comparison_ipw": cmp_ipw.to_dict(orient="records"),
    "observed_policy_value_test": factual_value_te,
    "a3_scores_used": False,
    "equity_group": "bw_tercile",
}
export_meta_path = ART_DIR / "export_meta.json"
export_meta_path.write_text(json.dumps(export_info, indent=2, default=str), encoding="utf-8")
print("wrote", export_meta_path)
print({k: export_info[k] for k in (
    "method_selected", "calibration_role", "evaluation_role",
    "threshold", "treat_rate_test", "qini_auc_test", "a3_scores_used",
    "evidence_status",
)})

# Smoke B3: cargar y predecir 3 filas, sin recalibrar
pol_loaded = load_policy(policy_path)
smoke = pol_loaded.predict(X=X_te.iloc[:3])
print("smoke decision:", smoke["decision"].tolist())
print("smoke scores  :", [round(float(s), 4) for s in smoke["uplift_score"]])


## 8. Discusión guiada (escribir en el entregable)

Completa en ½–1 página:

1. **Presupuesto:** si \(B\) baja de 0.20 a 0.10, ¿quién deja de ser tratado y por qué importa para welfare?
2. **Baseline outcome-only:** ¿en qué situación clínica/económica tratar al de *peor* outcome predicho es engañoso?
3. **Equidad:** ¿la política concentra tratamientos en algún tercil de `bw`? ¿Es eso indeseable *a priori* o reflejo de heterogeneidad real del efecto?
4. **Handoff B3:** ¿qué información del policy card no debería omitirse al servir una policy local?

### Respuesta de referencia (modelo — adaptar con tus números)

> Con menor \(B\) solo entran unidades con \(\hat\tau\) más alto; el welfare oracle por unidad tratada suele subir pero el impacto poblacional total puede caer.  
> Outcome-only ignora que el efecto del programa puede ser mayor en perfiles con outcome basal aceptable.  
> Diferencias de treat-rate por tercil de `bw` son una señal para auditoría, no una prueba de sesgo.  
> El card debe declarar datos sintéticos, \(B\), features, supuestos y non-claims.


## Sección de evaluación reservada

La edición pública termina antes de las celdas que requieren `truth`, `y0`, `y1` o `tau` simulados. La evaluación docente se ejecuta en un entorno separado. Continúa la interpretación con datos observados, incertidumbre y supuestos declarados.
